### IBM QAOA Parameter Setting Analysis ###
This notebook produces tables and generates plots to analyze real IBM Qunatum hardware data while running QAOA on ensembles of the MaxCut problem. Various training strategies are utilized to determine the optimal parameters (angles of QAOA) before running them on hardware. The idea is to create a resource-cost estimation for these different parameter setting strategies and suggest best practices.

In [1]:
%load_ext autoreload
%autoreload 2

# Path imports
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

# IBM QAOA package specific
from src.Processing import QAOAHardware, QAOATraining
from src.Processing import set_data_path
from src.Processing import load_problem_instance
from src.Appox_Ratio_Calc import get_minmax, extract_minmax_args, maxcut_approximation_ratio
from src.utils import is_empty_nested_list, parse_instance_name

# General useful Python libraries
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np  
from pathlib import Path
from IPython.display import display

# Add stochastic-benchmark src to path and import related libraries
sys.path.append('../../src')
import stochastic_benchmark as SB
import bootstrap
import interpolate
import stats
from utils_ws import *
from plotting import Plotting

#### Data Analysis for IBM Quantum Hardware Data ####

In [2]:
# Select graph to explore
graph_type = "heavy_hex"

# Set data path
data_dir = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data"
# Set instance path
inst_path = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/instances"
# Set minmax path for approximation ratio calculations later on (specific to evaluations pertaining the Maccut problem)
minmax_path = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/minmax_cuts"

# Set hardware data path to graph type
hardware_data_path = set_data_path(data_dir, True, False, graph_type)

# Set training data path to graph type
training_data_path = set_data_path(data_dir, False, True, graph_type)

# # Load instance path
# instance_path = load_problem_instance("heavy_hex")
# print(instance_path)

# Select hardware instance parameters to explore:
p_list = [5, 10] # eg. QAOA depths for heavy_hex
num_nodes = 144 # eg. Problem size
instance_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] # eg. instance index (last index only)

# Load heavy_hex instances for the above parameter combination
instance_paths_hardware = []

for p in p_list:
    for instance in instance_list:
        instance_paths_hardware.extend(QAOAHardware.locate_hardware_instance(hardware_data_path, 
        graph_type, str(instance), str(num_nodes), str(p))) # extend ensures flat list

In [3]:
all_instances_data_hardware = []

for instance_path in instance_paths_hardware:
    all_instances_data_hardware.extend(QAOAHardware.load_hardware_instance(instance_path))

# Expand each object into data columns
all_data_dicts_hardware = []

for object in all_instances_data_hardware:
    data_dict_hardware = {

        "instance_name" : object.instance_name,
        "QPU_time (s)" : object.QPU_time,
        "num_shots" : object.num_shots,
        "problem_class" : object.problem_class,
        "training_method" : object.training_method,
        "expected_energy" : object.expected_energy,
        "result_file" : object.result_file
    }
    all_data_dicts_hardware.append(data_dict_hardware)

In [4]:
# Creata a DataFrame to document the extracted raw hardware data  
df_hardware = pd.DataFrame(all_data_dicts_hardware)
print(df_hardware.head())

  instance_name  QPU_time (s)  num_shots problem_class training_method  \
0   000N144HH73             8       4096        maxcut        F_MPS_10   
1   000N144HH73             8       4096        maxcut         F_PP_10   
2   000N144HH73             8       4096        maxcut        I_MPS_10   
3   000N144HH73             8       4096        maxcut         I_PP_10   
4   000N144HH73             8       4096        maxcut    TQA_PP_opt_5   

   expected_energy                           result_file  
0        27.752374  000N144HH73_MC_F_MPS_optBD24_10.json  
1        50.809508    000N144HH73_MC_F_PP_optMW6_10.json  
2        24.198928  000N144HH73_MC_I_MPS_optBD24_10.json  
3        50.991603    000N144HH73_MC_I_PP_optMW6_10.json  
4        47.329537   000N144HH73_MC_TQA_PP_optMW6_5.json  


In [5]:
# Compute approximation ratio for each object and append to the DataFrame
minmax_cache = {} # Since the same instance is present in multiple rows of the DataFrame, just load and store it once
approximation_ratio = []

for _, row in df_hardware.iterrows():
    instance_name = row["instance_name"]
    energy = row["expected_energy"] 
    instance = instance_name[:3]
    if instance not in minmax_cache:
        minmax_instance_path = get_minmax(minmax_path, graph_type, instance, num_nodes) # Extract specific minmax_instance path
        minmax_cache[instance] = extract_minmax_args(minmax_instance_path) # Cache args for the speciic instance
    min_cut, max_cut, sum_weights = minmax_cache[instance] # Extract minmax args for the specific instance
    approximation_ratio.append(maxcut_approximation_ratio(min_cut, max_cut, sum_weights, energy)) # Evlaute approximation ratio for that row
df_hardware["approximation_ratio"] = approximation_ratio

display(df_hardware)    

,instance_name,QPU_time (s),num_shots,problem_class,training_method,expected_energy,result_file,approximation_ratio
0,000N144HH73,8,4096,maxcut,F_MPS_10,27.752374,000N144HH73_MC_F_MPS_optBD24_10.json,0.722389
1,000N144HH73,8,4096,maxcut,F_PP_10,50.809508,000N144HH73_MC_F_PP_optMW6_10.json,0.907153
2,000N144HH73,8,4096,maxcut,I_MPS_10,24.198928,000N144HH73_MC_I_MPS_optBD24_10.json,0.693914
3,000N144HH73,8,4096,maxcut,I_PP_10,50.991603,000N144HH73_MC_I_PP_optMW6_10.json,0.908612
4,000N144HH73,8,4096,maxcut,TQA_PP_opt_5,47.329537,000N144HH73_MC_TQA_PP_optMW6_5.json,0.879267
...,...,...,...,...,...,...,...,...
115,009N144HH73,11,4096,maxcut,F_PP_10,50.526331,009N144HH73_MC_F_PP_optMW6_10.json,0.889648
116,009N144HH73,11,4096,maxcut,I_MPS_10,33.728281,009N144HH73_MC_I_MPS_optBD24_10.json,0.760105
117,009N144HH73,11,4096,maxcut,I_PP_10,58.539871,009N144HH73_MC_I_PP_optMW6_10.json,0.951447
118,009N144HH73,11,4096,maxcut,TQA_PP_opt_10,54.351645,009N144HH73_MC_TQA_PP_optMW6_10.json,0.919148


In [6]:

parsed_cols = df_hardware["result_file"].apply(parse_instance_name)
df_hardware = pd.concat([df_hardware, parsed_cols], axis=1)

display(df_hardware.head())

,instance_name,QPU_time (s),num_shots,problem_class,training_method,expected_energy,result_file,approximation_ratio,instance_number,N,graph_type,graph_param,trainer_label,evaluator_label,p
0,000N144HH73,8,4096,maxcut,F_MPS_10,27.752374,000N144HH73_MC_F_MPS_optBD24_10.json,0.722389,000,144,heavy_hex,73,F,MPS,10
1,000N144HH73,8,4096,maxcut,F_PP_10,50.809508,000N144HH73_MC_F_PP_optMW6_10.json,0.907153,000,144,heavy_hex,73,F,PP,10
2,000N144HH73,8,4096,maxcut,I_MPS_10,24.198928,000N144HH73_MC_I_MPS_optBD24_10.json,0.693914,000,144,heavy_hex,73,I,MPS,10
3,000N144HH73,8,4096,maxcut,I_PP_10,50.991603,000N144HH73_MC_I_PP_optMW6_10.json,0.908612,000,144,heavy_hex,73,I,PP,10
4,000N144HH73,8,4096,maxcut,TQA_PP_opt_5,47.329537,000N144HH73_MC_TQA_PP_optMW6_5.json,0.879267,000,144,heavy_hex,73,TQA,PP,5


#### Data Analysis for IBM Quantum QAOA Parameter Training Data ####

In [7]:
instance_paths_training = []

for p in p_list:
    for instance in instance_list:
        instance_paths_training.extend(QAOATraining.locate_training_instance(training_data_path, graph_type, instance, 
        num_nodes, p, ER_probability=None, swap_layers=None, degree=None))

In [8]:
all_instances_data_training = []

for instance_path in instance_paths_training:
    obj = QAOATraining.load_training_instance(instance_path)
    all_instances_data_training.append(obj)

# Expand each object into data columns
all_data_dicts_training = []

for obj in all_instances_data_training:
    data_dict_training = {
        "instance_name": obj.instance_name,
        "pre_processing_time": obj.pre_processing_time,
        "pre_processor_name": obj.pre_processor_name,
        "train_duration_per_iter": obj.train_duration_per_iter,
        "expected_energy_per_iter": obj.expected_energy_per_iter,
        "trainer_name_per_iter": obj.trainer_name_per_iter,
        "train_duration_per_layer": obj.train_duration_per_layer,
        "expected_energy_per_layer": obj.expected_energy_per_layer,
        "trainer_name_per_layer": obj.trainer_name_per_layer,
        "num_depth_iter": obj.num_depth_iter,
    }
    all_data_dicts_training.append(data_dict_training)

In [9]:
# Creata a DataFrame to document the raw extracted training data  
df_training = pd.DataFrame(all_data_dicts_training)
print(df_training.head())

                              instance_name  pre_processing_time  \
0       000N144HH73_MC_TQA_PP_optMW6_5.json                  NaN   
1     000N144HH73_MC_TQA_MPS_optBD24_5.json           289.975533   
2        000N144HH73_MC_LR_PP_optMW6_5.json           346.529271   
3  000N144HH73_MC_LR_MPSAer_opt_DB24_5.json           407.179045   
4       001N144HH73_MC_TQA_PP_optMW6_5.json                  NaN   

  pre_processor_name                  train_duration_per_iter  \
0               None  [51.947216510772705, 317.8320360183716]   
1          SATMapper  [168.91438794136047, 429.0275182723999]   
2          SATMapper                      [79.94285893440247]   
3          SATMapper                      [928.9998672008514]   
4               None   [45.84358072280884, 234.5539755821228]   

                  expected_energy_per_iter  \
0  [41.77672806116267, 47.329536787543674]   
1  [23.959557152217407, 27.55875064098228]   
2                      [41.77546875464326]   
3               

In [10]:
# Clean the DataFrame and report NaN to missing values
df_training = df_training.map(
    lambda x: np.nan if (x is None or x == [] or x == {} or is_empty_nested_list(x)) else x
)
print(df_training.loc[40, :])

instance_name                             000N144HH73_MC_F_MPS_optBD24_10.json
pre_processing_time                                                 279.515143
pre_processor_name                                                   SATMapper
train_duration_per_iter      [8.646271467208862, 1.5646839141845703, 4766.8...
expected_energy_per_iter     [29.890184793139888, 30.783545572306778, 22.64...
trainer_name_per_iter        [{'trainer_name': 'DepthOneScanTrainer', 'eval...
train_duration_per_layer     [[], [], [225.6993420124054, 332.2614352703094...
expected_energy_per_layer    [[], [], [23.987088354636604, 26.9366705694871...
trainer_name_per_layer       [[], [], [{'trainer_name': 'ScipyTrainer', 'ev...
num_depth_iter                          [[], [], [2, 3, 4, 5, 6, 7, 8, 9, 10]]
Name: 40, dtype: object


In [11]:
# This cell basically cleans up the DataFrame from a nested structure to a flat structure for better analysis

flat_rows = []

for _, row in df_training.iterrows():
    inst = row["instance_name"]

    outer_dur = row["train_duration_per_iter"]
    outer_eng = row["expected_energy_per_iter"]
    outer_trn = row["trainer_name_per_iter"]

    inner_dur   = row["train_duration_per_layer"]
    inner_eng   = row["expected_energy_per_layer"]
    inner_trn   = row["trainer_name_per_layer"]
    inner_depth = row["num_depth_iter"]

    if not isinstance(outer_dur, list):
        continue

    for i in range(len(outer_dur)):
        trn = outer_trn[i] if isinstance(outer_trn, list) and i < len(outer_trn) and isinstance(outer_trn[i], dict) else {}
        min_args = trn.get("minimize_args") if isinstance(trn.get("minimize_args"), dict) else {}
        opts = min_args.get("options") if isinstance(min_args.get("options"), dict) else {}

        flat_rows.append({
            "instance": inst,
            "level": "outer",
            "iteration": i,
            "duration": outer_dur[i] if i < len(outer_dur) else np.nan,
            "energy": outer_eng[i] if isinstance(outer_eng, list) and i < len(outer_eng) else np.nan,
            "trainer": trn.get("trainer_name"),
            "method": min_args.get("method"),
            "maxiter": opts.get("maxiter"),
            "depth_step": np.nan,
        })

        # ----- inner -----
        dur_i   = inner_dur[i]   if isinstance(inner_dur, list)   and i < len(inner_dur)   and isinstance(inner_dur[i], list)   else []
        eng_i   = inner_eng[i]   if isinstance(inner_eng, list)   and i < len(inner_eng)   and isinstance(inner_eng[i], list)   else []
        trn_i   = inner_trn[i]   if isinstance(inner_trn, list)   and i < len(inner_trn)   and isinstance(inner_trn[i], list)   else []
        depth_i = inner_depth[i] if isinstance(inner_depth, list) and i < len(inner_depth) and isinstance(inner_depth[i], list) else []

        for dur, eng, trn2, depth in zip(dur_i, eng_i, trn_i, depth_i):
            trn2 = trn2 if isinstance(trn2, dict) else {}
            min_args2 = trn2.get("minimize_args") if isinstance(trn2.get("minimize_args"), dict) else {}
            opts2 = min_args2.get("options") if isinstance(min_args2.get("options"), dict) else {}

            flat_rows.append({
                "instance": inst,
                "level": "inner",
                "iteration": i,
                "depth_step": depth,
                "duration": dur,
                "energy": eng,
                "trainer": trn2.get("trainer_name"),
                "method": min_args2.get("method"),
                "maxiter": opts2.get("maxiter"),
            })

df_flat = pd.DataFrame(flat_rows)

In [12]:
# Compute approximation ratio for each object and append to the DataFrame. Use the same cache created for hardware data
approximation_ratio_training = []

for _, row in df_flat.iterrows():
    instance_name = row["instance"]
    energy = row["energy"] 
    instance = instance_name[:3]
    if instance not in minmax_cache:
        minmax_instance_path = get_minmax(minmax_path, graph_type, instance, num_nodes) # Extract specific minmax_instance path
        minmax_cache[instance] = extract_minmax_args(minmax_instance_path) # Cache args for the speciic instance
    min_cut, max_cut, sum_weights = minmax_cache[instance] # Extract minmax args for the specific instance
    approximation_ratio_training.append(maxcut_approximation_ratio(min_cut, max_cut, sum_weights, energy)) # Evlaute approximation ratio for that row
df_flat["approximation_ratio"] = approximation_ratio_training   

In [13]:
# Extract instance specific details
parsed_cols = df_flat["instance"].apply(parse_instance_name)
df_flat = pd.concat([parsed_cols, df_flat], axis=1)

display(df_flat.head())

,instance_number,N,graph_type,graph_param,trainer_label,evaluator_label,p,instance,level,iteration,duration,energy,trainer,method,maxiter,depth_step,approximation_ratio
0,000,144,heavy_hex,73,TQA,PP,5,000N144HH73_MC_TQA_PP_optMW6_5.json,outer,0,51.947217,41.776728,TQATrainer,None,NaN,NaN,0.834771
1,000,144,heavy_hex,73,TQA,PP,5,000N144HH73_MC_TQA_PP_optMW6_5.json,outer,1,317.832036,47.329537,ScipyTrainer,COBYLA,100.0,NaN,0.879267
2,000,144,heavy_hex,73,TQA,MPS,5,000N144HH73_MC_TQA_MPS_optBD24_5.json,outer,0,168.914388,23.959557,TQATrainer,None,NaN,NaN,0.691996
3,000,144,heavy_hex,73,TQA,MPS,5,000N144HH73_MC_TQA_MPS_optBD24_5.json,outer,1,429.027518,27.558751,ScipyTrainer,COBYLA,50.0,NaN,0.720837
4,000,144,heavy_hex,73,LR,PP,5,000N144HH73_MC_LR_PP_optMW6_5.json,outer,0,79.942859,41.775469,TQATrainer,None,NaN,NaN,0.834761


#### Integrating with Stochastic Benchmark Framework ####